In [ ]:
# Import libraries and init EE
import pandas as pd
import ee
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# ee.Authenticate()
ee.Initialize(project='qsair-463811')

In [ ]:
# Define Region of Interest(Area near Iganga)
region = ee.Geometry.Polygon([
    [ [32.525, 0.410], [32.525, 0.425],
      [32.550, 0.425], [32.550, 0.410] ]
])


In [ ]:
# Get S2 Imagery
s2_image_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                       .filterBounds(region)
                       .filterDate('2021-01-01', '2021-12-31')
                       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))


In [ ]:
# Create Composite and Calculate Indices
composite = s2_image_collection.median()
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
mndwi = composite.normalizedDifference(['B3', 'B11']).rename('MNDWI')


In [ ]:
# Add Indices to Composite
input_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI', 'MNDWI']
composite_with_bands = composite.addBands([ndvi, mndwi])


In [ ]:
# Load ground truth data
samples = ee.FeatureCollection('projects/qsair-463811/assets/ground_truth')


In [ ]:
# Sample regions
sampled_data = composite_with_bands.sampleRegions(
    collection=samples,
    properties=['crop_type'],
    scale=10
)


In [ ]:
# Define Function to Convert EE Collection to Pandas DataFrame
def ee_to_pandas(ee_collection):
    """Converts an Earth Engine FeatureCollection to a Pandas DataFrame."""
    feature_list = ee_collection.getInfo()['features']
    data = [feature['properties'] for feature in feature_list]
    return pd.DataFrame(data)

# Convert to Pandas Dataframe
df = ee_to_pandas(sampled_data)


In [ ]:
# Prepare Data for Scikit-Learn
X = df[input_bands]
y = df['crop_type']


In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:
# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=500, random_state=42)
model.fit(X_train, y_train)


In [ ]:
# Model Evaluation
from sklearn.metrics import classification_report
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))
